In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats as stats
from scipy.stats import pearsonr
from dash import html, dcc, Input, Output, Dash
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))
from baseVR.base_functionality import init_import_paths
init_import_paths() 

from CustomLogger import CustomLogger as Logger
from analytics_processing import analytics

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args, fullfnames2snames
from dashsrc.plot_components.plots import plot_TrackFiringRate
from dashsrc.plot_components.plots import plot_unit_fr_stability

# from ..dashsrc.plot_components.plots import plot_TrackFiringRate
# from ..dashsrc.plot_components.plot_wrappers import wrapper_TrackFiringRate


In [2]:
Logger().init_logger(None, None, logging_level="DEBUG")
animal_ids = [6]
paradigm = [1100]
session_range = [1,33]
session_ids = None
normalize = True
smooth = False
excl_session_names =  ['2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min', '2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min', '2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min', '2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min'] # ignore short sessions (10, 24,25)

session_dirs = sessionlist_fullfnames_from_args(paradigm, animal_ids, session_ids, excl_session_names=excl_session_names)[0]
session_names= fullfnames2snames(session_dirs)

2026-02-17 16:32:25,413|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-02-17 16:32:25,770|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min.hdf5 excluded
2026-02-17 16:32:25,931|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-17 16:32:26,131|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-17 16:32:26,149|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min.hdf5 excluded
2026-02-17 16:32:26,435|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 30 sessions.
2026-02-17 16:32:26,438|DEBUG|70427|sessions_from_nas_parsing|sessionl

In [3]:
# sess_ids_beh = behav.index.get_level_values("session_id").unique()
# sess_ids_ens = pd.Index(ens_proj["session_id"].unique())
# sess_ids = [s for s in sess_ids_beh if s in set(sess_ids_ens)]

# all_sessions = []

# for s_id in sess_ids:
#     beh_s = behav.xs(s_id, level="session_id").copy()
#     ens_s = ens_proj.loc[ens_proj["session_id"] == s_id].copy()

#     # recover interval index for this session only
#     ens_s.index = pd.IntervalIndex.from_arrays(
#         ens_s.pop("from_ephys_timestamp"),
#         ens_s.pop("to_ephys_timestamp"),
#     )

#     ens_s = ens_s.drop(columns=["session_id"])

#     def time_bin_avg(posbin_data: pd.DataFrame) -> pd.DataFrame:
#         # remove rows that cannot define intervals
#         # posbin_data = posbin_data.dropna(
#         #     subset=["posbin_from_ephys_timestamp", "posbin_to_ephys_timestamp"]
#         # )
#         if posbin_data.empty:
#             return pd.DataFrame()

#         # convert to real NumPy ints (avoids nullable Int64 masked arrays)
#         from_posbin_t = posbin_data["posbin_from_ephys_timestamp"].to_numpy(dtype="int64")
#         to_posbin_t   = posbin_data["posbin_to_ephys_timestamp"].to_numpy(dtype="int64")

#         interval = pd.IntervalIndex.from_arrays(
#             from_posbin_t - 40_000,
#             to_posbin_t + 40_000,
#             closed="both",
#         )

#         mid = ens_s.index.mid

#         print("mid range:", mid.min(), mid.max())
#         print("interval range:", interval.left.min(), interval.right.max())

#         # assign each 40ms ensemble bin midpoint to a trial in this position bin
#         assigned_bin = pd.cut(
#             ens_s.index.mid,
#             bins=interval,
#             labels=posbin_data.trial_id[:-1],
#         )
#         trials_exist_mask = (assigned_bin.value_counts() != 0).values

#         # if nothing overlaps, return empty
#         if assigned_bin.notna().sum() == 0:
#             return pd.DataFrame()

#         # keep only overlapping bins
#         trial_proj = ens_s.loc[assigned_bin.notna()].copy().astype(np.float32)
#         trial_proj["posbin_t_edges"] = assigned_bin[assigned_bin.notna()]

#         posbin_trial_wise = trial_proj.groupby("posbin_t_edges", observed=True).mean()

#         # align metadata to the set of trials that actually exist
#         # vc = assigned_bin.value_counts()
#         # trials_exist = vc.index.to_numpy()
#         # trials_exist_mask = posbin_data["trial_id"].isin(trials_exist).to_numpy()

#         posbin_trial_wise["cue"] = posbin_data.loc[trials_exist_mask, "cue"].to_numpy()
#         posbin_trial_wise["trial_outcome"] = posbin_data.loc[trials_exist_mask, "trial_outcome"].to_numpy()
#         posbin_trial_wise["choice_R1"] = posbin_data.loc[trials_exist_mask, "choice_R1"].to_numpy()
#         posbin_trial_wise["choice_R2"] = posbin_data.loc[trials_exist_mask, "choice_R2"].to_numpy()
#         posbin_trial_wise["bin_length"] = interval.length[trials_exist_mask] / 1e6

#         posbin_trial_wise.index = posbin_data.loc[trials_exist_mask, "trial_id"].to_numpy()
#         return posbin_trial_wise

#     # group within session only
#     sess_out = beh_s.groupby("from_position_bin").apply(time_bin_avg)
#     if isinstance(sess_out, pd.DataFrame) and len(sess_out) > 0:
#         sess_out.index = sess_out.index.rename(["from_position_bin", "trial_id"])
#         sess_out = sess_out.reset_index(drop=False)
#         sess_out["session_id"] = s_id
#         all_sessions.append(sess_out)

# all_sessions


In [4]:
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)

2026-02-17 16:32:26,469|DEBUG|70427|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-17 16:32:26,754|DEBUG|70427|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-17 16:32:26,755|DEBUG|70427|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-17 16:32:26,768|INFO|70427|analytics|get_analytics
	Analytic `FiringRate40msHz` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-17 16:32:26,768|DEBUG|70427|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min.hdf5

In [5]:
# firing rates and behavior data
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr_track = analytics.get_analytics('FiringRateTrackwiseHz', session_names=session_names)
# fr_z_scored  = analytics.get_analytics('FiringRate40msZ', session_names=session_names)
fr_z_all_sess = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))
# behav = analytics.get_analytics('BehaviorTrackwise', session_names=session_names)
# behav.index = behav.index.droplevel(('animal_id', 'paradigm_id', 'entry_id'))

2026-02-17 16:32:37,067|DEBUG|70427|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-17 16:32:37,111|DEBUG|70427|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-17 16:32:37,113|DEBUG|70427|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_15-01') 2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min.hdf5
/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min
2026-02-17 16:32:37,115|INFO|70427|analytics|get_analytics
	Analytic `FiringRate40msHz` does not exist for (1100, 6, '2024-11-14_15-01'), compute first, or check for typo
2026-02-17 16:32:37,119|DEBUG|70427|analytics|get_analytics
	Processing FiringRate40msHz, (1100, 6, '2024-11-14_16-40') 2024-11-14_16-40_rYL006_P1100_LinearTrackStop_21min.hdf5

In [6]:
# ensamble related data
ensambles = analytics.get_analytics('ConcatenatedEnsambles40ms', session_names=session_names)
ens_data = analytics.get_analytics('TrackwiseEnsembleProj', session_names=session_names)
ensamble_proj = analytics.get_analytics('ConcatenatedEnsambleProj40ms', session_names=session_names)#.drop("to_ephys_timestamp", axis=1)
ensamble_proj.set_index(['session_id'], inplace = True)
ens_data = ens_data.set_index(['session_id', 'trial_id']).sort_index()

2026-02-17 16:32:54,354|DEBUG|70427|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-17 16:32:54,395|DEBUG|70427|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-17 16:32:54,410|DEBUG|70427|analytics|get_analytics
	Processing ConcatenatedEnsambles40ms, for n=30 sessions


/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min/../../animal_analytics/ConcatenatedEnsambles40ms.parquet


2026-02-17 16:32:54,705|INFO|70427|analytics|get_analytics
	Loaded animal-level analytic `ConcatenatedEnsambles40ms` last modified on 2026-02-05T15:17:08.109814
2026-02-17 16:32:54,707|DEBUG|70427|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-17 16:32:54,719|DEBUG|70427|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-17 16:32:54,724|DEBUG|70427|analytics|get_analytics
	Processing TrackwiseEnsembleProj, for n=30 sessions


/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min/../../animal_analytics/TrackwiseEnsembleProj.parquet


2026-02-17 16:33:19,058|INFO|70427|analytics|get_analytics
	Loaded animal-level analytic `TrackwiseEnsembleProj` last modified on 2026-02-17T13:13:21.158725
2026-02-17 16:33:19,061|DEBUG|70427|sessions_from_nas_parsing|sessionnames2fullfnames
	Inferring NAS paths for list of session names...
2026-02-17 16:33:19,346|DEBUG|70427|sessions_from_nas_parsing|sessionlist_fullfnames_from_args
	Paradigm_ids: None, animal_ids: None, session_ids: None, from_date: None, to_date: None
	Merging 30 sessions

2026-02-17 16:33:19,349|DEBUG|70427|analytics|get_analytics
	Processing ConcatenatedEnsambleProj40ms, for n=30 sessions


/Volumes/large/BMI/VirtualReality/SpatialSequenceLearning/RUN_rYL006/rYL006_P1100/2024-11-14_15-01_rYL006_P1100_LinearTrackStop_30min/../../animal_analytics/ConcatenatedEnsambleProj40ms.parquet


2026-02-17 16:34:27,784|INFO|70427|analytics|get_analytics
	Loaded animal-level analytic `ConcatenatedEnsambleProj40ms` last modified on 2026-02-05T15:20:02.204022


In [7]:
# df = ens_data.reset_index() 
sessions_keep = [8, 9, 11, 12, 13, 14, 15, 16] #C1 strategy
# sessions_keep = [15, 16, 17, 18, 19, 20, 21, 22, 23] #C2 strategy
#sessions_keep = [23] 



# keep only expert trials (C1 cue -> choice_R1 = True, choice_R2 = False; C2 cue -> choice_R1 = False, choice_R2 = True)
ens_filt = ens_data.loc[ens_data.index.get_level_values("session_id").isin(sessions_keep)]

ens_filt = ens_data.copy()

# expert1 = ens_filt[ # Keep only expert trials
#         (ens_filt["choice_R1"] == 1) &
#         (ens_filt["choice_R2"] == 0) &
#         (ens_filt["cue"] == 1)
#     ]

# expert2 = ens_filt[ # Keep only expert trials
#         (ens_filt["choice_R1"] == 0) &
#         (ens_filt["choice_R2"] == 1) &
#         (ens_filt["cue"] == 2)
#     ]

expert1 = ens_filt[ # Keep only non expert trials
        (ens_filt["choice_R1"] == 0) &
        (ens_filt["cue"] == 1)
    ]

expert2 = ens_filt[ # Keep only non expert trials
        (ens_filt["choice_R2"] == 0) &
        (ens_filt["choice_R1"] == 1) &
        (ens_filt["cue"] == 2)
    ]



ens_filt = pd.concat([expert1, expert2])


trial_avg = (
    ens_filt # .reset_index()
    .groupby(["from_position_bin", "cue"], as_index=False)["Assembly012"]
    .mean()
    .rename(columns={"Assembly012": "Assembly012_mean"})
)

# 2) Keep only cue 1 and 2
trial_avg = trial_avg[trial_avg["cue"].isin([1, 2])]

# 3) Ensure one bar per discrete trial_id (no numeric binning)
trial_avg["from_position_bin"] = trial_avg["from_position_bin"].astype(str)

cue1 = trial_avg[trial_avg["cue"] == 1]
cue2 = trial_avg[trial_avg["cue"] == 2]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=cue1["from_position_bin"],
        y=cue1["Assembly012_mean"],
        name="cue 1",
        marker_color = 'orange',
        opacity=0.7,
    )
)


fig.add_trace(
    go.Bar(
        x=cue2["from_position_bin"],
        y=cue2["Assembly012_mean"],
        name="cue 2",
        marker_color = 'purple',
        opacity=0.4,
    )
)

fig.update_layout(
    # title = f"Average Ensemble Activity trackwise - Sessions {min(sessions_keep)} to {max(sessions_keep)}",
    title = f"Average Ensemble Activity trackwise - All Sessions (Expert only)",
    barmode="overlay",  # use "group" for side-by-side bars
    xaxis_title="from_position_bin",
    yaxis_title="Avg(Assembly012) per position bin",
    bargap=0.05,
)

fig.update_xaxes(type="linear")
fig.update_yaxes(range=[min(trial_avg["Assembly012_mean"]), 1])

fig.add_vrect(x0=-80, x1=25,  fillcolor="orange", opacity=0.1, layer="below", line_width=0)
fig.add_vrect(x0=50,  x1=110, fillcolor="grey",   opacity=0.1, layer="below", line_width=0)
fig.add_vrect(x0=170, x1=230, fillcolor="grey",   opacity=0.1, layer="below", line_width=0)


fig.show()



In [8]:
# import matplotlib.pyplot as plt
# plt.close('all')
# for i in range(ens_data.shape[1]):
#     plt.figure()
#     plt.hist(np.clip(ens_data.iloc[:, i], -4, 7), bins=100, color='steelblue', alpha=0.7, range=(-4, 7))
#     plt.title(f'Ensamble {i} projection - overall activation strength')
#     plt.show()

In [9]:
#ordering neurons by weight in ens
neurons_ordered = abs(ensambles['Assembly012']).sort_values(ascending=True).index
neurons_ordered

Index([ 4, 29, 43, 76, 10, 58, 25, 27, 13, 37, 59, 46, 47, 75, 57, 19, 20, 34,
       67,  5, 42, 73, 66, 60, 61, 70, 49, 28, 18, 30, 17, 63, 41,  3, 54, 52,
       33, 39, 16,  9, 26, 69,  0, 44, 40,  6, 38, 12, 45, 71, 55, 32, 51, 31,
        1, 64, 53, 21, 56, 74, 72,  2, 35, 65,  7, 62,  8, 36, 68, 23, 14, 24,
       22, 48, 15, 11, 50],
      dtype='int64')

In [10]:
meta_data = {}
meta_data['SpikeClusterMetadata'] = analytics.get_analytics('SpikeClusterMetadata', mode='set',
                                                      #  columns = cols,
                                                       paradigm_ids=paradigm,
                                                       animal_ids=animal_ids,
                                                       excl_session_names=excl_session_names,
                                                       session_ids=session_ids)


meta_data['SpikeClusterMetadata']

2026-02-17 16:34:28,620|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-02-17 16:34:29,135|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min.hdf5 excluded
2026-02-17 16:34:29,394|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-17 16:34:29,703|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-17 16:34:29,734|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min.hdf5 excluded
2026-02-17 16:34:30,148|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 30 sessions.
2026-02-17 16:34:30,150|DEBUG|70427|sessions_from_nas_parsing|sessionl

cluster_id cluster_type  \
paradigm_id animal_id session_id       entry_id                            
1100        6         2024-11-14_16-40 0                  1       single   
                                       1                  2        3-ISI   
                                       2                  3       single   
                                       3                  4        5-ISI   
                                       4                  5       single   
...                                                     ...          ...   
                      2025-01-27_13-39 72                73       single   
                                       73                74       single   
                                       74                75       single   
                                       75                76       single   
                                       76                77       single   

                                                 unit_count  cluster_channel  \
paradigm_id animal_id session_id       entry_id                                
1100        6         2024-11-14_16-40 0            44619.0            295.0   
                                       1           208600.0            299.0   
                                       2            11050.0            297.0   
                                       3            18939.0            309.0   
                                       4            23034.0            311.0   
...                                                     ...              ...   
                      2025-01-27_13-39 72           28147.0            150.0   
                                       73          585584.0            151.0   
                                       74          370562.0            154.0   
                                       75          272688.0            156.0   
                                       76          200568.0            158.0   

                                                 cluster_id_ssbatch  unit_snr  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0                          1  6.475497   
                                       1                          2  4.558548   
                                       2                          3  4.637581   
                                       3                          4  5.504494   
                                       4                          5  5.848962   
...                                                             ...       ...   
                      2025-01-27_13-39 72                        56  6.038227   
                                       73                        57  7.484550   
                                       74                        58  6.215347   
                                       75                        59  6.829296   
                                       76                        60  6.174706   

                                                  unit_Vpp  unit_isi_ratio  \
paradigm_id animal_id session_id       entry_id                              
1100        6         2024-11-14_16-40 0         61.824997        0.173228   
                                       1         53.466000        0.722572   
                                       2         55.282997        0.517730   
                                       3         53.279999        0.492537   
                                       4         49.151997        0.198254   
...                                                    ...             ...   
                      2025-01-27_13-39 72        77.212997        0.156766   
                                       73        81.588997        0.079114   
                                       74        65.533997        0.094184   
                                       75        68.143005        0.070829   
                             

In [11]:
meta_data = {}
meta_data['SpikeClusterMetadata'] = analytics.get_analytics('SpikeClusterMetadata', mode='set',
                                                      #  columns = cols,
                                                       paradigm_ids=paradigm,
                                                       animal_ids=animal_ids,
                                                       excl_session_names=excl_session_names,
                                                       session_ids=session_ids)

# # HPC
# meta_data['SpikeClusterMetadata'] = meta_data['SpikeClusterMetadata'][meta_data['SpikeClusterMetadata'].cluster_id<=20]
# # mPFC
# meta_data['SpikeClusterMetadata'] = meta_data['SpikeClusterMetadata'][meta_data['SpikeClusterMetadata'].cluster_id<20]
# meta_data['SpikeClusterMetadata']

2026-02-17 16:34:36,984|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Searching NAS for applicable sessions...
2026-02-17 16:34:36,994|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min.hdf5 excluded
2026-02-17 16:34:36,995|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2024-12-11_17-42_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-17 16:34:36,996|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min.hdf5 excluded
2026-02-17 16:34:36,997|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	Session 2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min.hdf5 excluded
2026-02-17 16:34:36,998|DEBUG|70427|sessions_from_nas_parsing|get_sessionlist_fullfnames
	For paradigms [1100], animals [6], found 30 sessions.
2026-02-17 16:34:36,999|DEBUG|70427|sessions_from_nas_parsing|sessionl

In [12]:
neurons_ordered

Index([ 4, 29, 43, 76, 10, 58, 25, 27, 13, 37, 59, 46, 47, 75, 57, 19, 20, 34,
       67,  5, 42, 73, 66, 60, 61, 70, 49, 28, 18, 30, 17, 63, 41,  3, 54, 52,
       33, 39, 16,  9, 26, 69,  0, 44, 40,  6, 38, 12, 45, 71, 55, 32, 51, 31,
        1, 64, 53, 21, 56, 74, 72,  2, 35, 65,  7, 62,  8, 36, 68, 23, 14, 24,
       22, 48, 15, 11, 50],
      dtype='int64')

In [13]:
i = "Assembly012"
fig_heat = plot_unit_fr_stability.render_plot_heatmap(meta_data["SpikeClusterMetadata"])
heatmap = fig_heat.data[0]

current_labels = np.array(fig_heat.layout.yaxis.ticktext, dtype=str)

desired_order = np.array( neurons_ordered[::], dtype=str)
desired_order = [lab for lab in desired_order if lab in current_labels]

label_to_row = {lab: j for j, lab in enumerate(current_labels)}
row_indices = [label_to_row[lab] for lab in desired_order]

z = np.array(heatmap.z)
z_reordered = z[row_indices, :]
xlabels = np.array(fig_heat.layout.xaxis.ticktext, dtype=str)

# subplot creation
fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.3, 0.7],
    subplot_titles=("Ensemble 12", "Average Firing Rate per Neuron/Session"),
)

# Left plot
ens_idx_str = (ensambles.index+1).astype(str)
weights = pd.Series(ensambles[i].values, index=ens_idx_str)

# HPC mPC split
labels_hp   = ens_idx_str[:20]
labels_mpfc = ens_idx_str[20:]

# reorder those labels according to desired_order (same as heatmap y)
labels_hp_ordered   = [lab for lab in desired_order if lab in labels_hp]
labels_mpfc_ordered = [lab for lab in desired_order if lab in labels_mpfc]

w_hp   = weights.loc[labels_hp_ordered].to_numpy()
w_mpfc = weights.loc[labels_mpfc_ordered].to_numpy()

def make_stem_lines(x_vals, y_labels, color):
    """Horizontal 'stems' at categorical y positions."""
    xs, ys = [], []
    for x, ylab in zip(x_vals, y_labels):
        xs.extend([0, x, None])
        ys.extend([ylab, ylab, None])
    return go.Scatter(
        x=xs,
        y=ys,
        mode="lines",
        line=dict(color=color),
        showlegend=False,
        hoverinfo="skip",
    )

hp_stems = make_stem_lines(w_hp,   labels_hp_ordered,   color="blue")
mpfc_stems = make_stem_lines(w_mpfc, labels_mpfc_ordered, color="magenta")

hp_markers = go.Scatter(
    x=w_hp,
    y=labels_hp_ordered,
    mode="markers",
    name=f"{i}, HP",
    marker=dict(color="blue", symbol="circle"),
)

mpfc_markers = go.Scatter(
    x=w_mpfc,
    y=labels_mpfc_ordered,
    mode="markers",
    name=f"{i}, mPFC",
    marker=dict(color="magenta", symbol="circle"),
)

thr = 0.2
mask = np.abs(weights.values) > thr
x_high = weights.values[mask]
y_high = ens_idx_str[mask]  

highlight = go.Scatter(
    x=x_high,
    y=y_high,
    mode="markers",
    name="Otsu thresh.",
    marker=dict(
        size=10,
        color="rgba(0,0,0,0)",
        line=dict(color="black", width=2),
        symbol="circle",
    ),
)

fig.add_trace(hp_stems,     row=1, col=1)
fig.add_trace(mpfc_stems,   row=1, col=1)
fig.add_trace(hp_markers,   row=1, col=1)
fig.add_trace(mpfc_markers, row=1, col=1)
fig.add_trace(highlight,    row=1, col=1)

fig.update_xaxes(
    title_text="Weight",
    range=[-0.45, 0.45],
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Neurons",
    categoryorder="array",
    categoryarray=desired_order,
    dtick=1,
    row=1,
    col=1,
)

fig.add_vline(
    x=0,
    line_width=1,
    line_color="black",
    row=1,
    col=1,
)

# Heatmap
heatmap_trace = go.Heatmap(
    z=z_reordered,
    x=xlabels,
    y=desired_order,
    colorscale=heatmap.colorscale,
    zmin=heatmap.zmin,
    zmax=heatmap.zmax,
    colorbar=dict(
        title=heatmap.colorbar.title.text,
        tickvals=heatmap.colorbar.tickvals,
        ticktext=heatmap.colorbar.ticktext,
        tickmode=heatmap.colorbar.tickmode,
        len=heatmap.colorbar.len,
        thickness=heatmap.colorbar.thickness,
        tickfont=dict(size=heatmap.colorbar.tickfont.size),
    ),
)

fig.add_trace(heatmap_trace, row=1, col=2)

fig.update_xaxes(
    title_text=fig_heat.layout.xaxis.title.text,
    row=1,
    col=2,
)

fig.update_yaxes(
    title_text=fig_heat.layout.yaxis.title.text,
    categoryorder="array",
    dtick=1,
    categoryarray=desired_order,
    row=1,
    col=2,
)

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=800,
    legend=dict(font=dict(size=6)),
)

fig.show()


In [14]:
session_used = '2024-12-03_16-23'
neuron_data_s15 = fr.xs(session_used, level=2)

# Filter Unit columns only
unit_cols = [col for col in neuron_data_s15.columns if isinstance(col, str) and col.startswith('Unit')]
neuron_data_s15 = neuron_data_s15[unit_cols]

# Compute correlation matrix (neuron vs neuron)
corr_matrix = neuron_data_s15.corr()

# Create ordering based on reversed neurons_ordered
# neurons_ordered contains indices 0-76, map to Unit column names
reversed_order = neurons_ordered[::-1]
unit_order = [f'Unit{i+1:04d}' for i in reversed_order if f'Unit{i+1:04d}' in corr_matrix.columns]

# Reorder the correlation matrix
corr_matrix_ordered = corr_matrix.loc[unit_order, unit_order]

# Create heatmap
fig = px.imshow(
    corr_matrix_ordered,
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    labels=dict(x="Neurons", y="Neurons", color="Pearson r"),
    title=f"Neuron-Neuron Pearson Correlation Matrix (Session {session_used}, Assembly012)"
)

fig.update_xaxes(tickangle=90, tickfont=dict(size=6))
fig.update_yaxes(tickfont=dict(size=6))
fig.update_layout(width=800, height=800)

fig.show()

In [49]:
# selecting only the rows of fr_track where column from_position_bin  is between 148 and 151
fr_track_selected = fr_track[(fr_track['from_position_bin'] >= (-95)) & (fr_track['from_position_bin'] <= 10)] # cue zone

# further filter to rows where cue == 1
# fr_track_selected = fr_track_selected[fr_track_selected['cue'] == 1]
# fr_track_selected = fr_track_selected[fr_track_selected['choice_R2'] == False]

In [16]:
# session_id = fr_track_selected.index.get_level_values("session_id").unique()
# session_id
# x = fr_track_selected.xs(session_id, level=2)
# x
session_ids_all = fr_track_selected.index.get_level_values("session_id").unique()
session_ids_all = pd.to_datetime(session_ids_all, format="%Y-%m-%d_%H-%M")
session_ids_all

DatetimeIndex(['2024-11-14 16:40:00', '2024-11-15 15:48:00',
               '2024-11-20 17:46:00', '2024-11-21 17:22:00',
               '2024-11-25 16:25:00', '2024-11-26 16:39:00',
               '2024-11-28 17:41:00', '2024-12-02 16:09:00',
               '2024-12-03 16:23:00', '2024-12-04 18:06:00',
               '2024-12-06 16:49:00', '2024-12-09 17:45:00',
               '2024-12-10 17:20:00', '2024-12-12 16:13:00',
               '2024-12-13 17:10:00', '2025-01-14 18:08:00',
               '2025-01-15 17:18:00', '2025-01-16 17:47:00',
               '2025-01-17 16:55:00', '2025-01-23 16:48:00',
               '2025-01-24 12:24:00', '2025-01-24 19:37:00',
               '2025-01-25 21:29:00', '2025-01-26 21:48:00',
               '2025-01-27 13:39:00'],
              dtype='datetime64[ns]', name='session_id', freq=None)

In [50]:
start = pd.Timestamp("2024-12-14")
end   = pd.Timestamp("2025-01-26")

# set top x neurons and select sessions
top_x_neurons = neurons_ordered[-20:]

session_ids_all = fr_track_selected.index.get_level_values("session_id").unique()
dt_index = pd.to_datetime(session_ids_all, format="%Y-%m-%d_%H-%M")

# boolean selection
mask = (dt_index >= start) & (dt_index <= end)
selected = session_ids_all[mask]



n_sessions = len(selected)

n_cols = 4 # int(np.ceil(np.sqrt(n_sessions)))
n_rows = 2 # int(np.ceil(n_sessions / n_cols))

reversed_order_topx = top_x_neurons[::-1]

# Create subplots
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    #subplot_titles=[f'S{s}' for s in selected],
    horizontal_spacing=0.02,
    vertical_spacing=0.03,
)

# Compute correlation matrix for each session (only top 10 neurons)
for idx, session_id in enumerate(selected):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    
    try:
        neuron_data_session = fr_track_selected.xs(session_id, level=2)
        
        # Filter to Unit columns only
        unit_cols = [c for c in neuron_data_session.columns if isinstance(c, str) and c.startswith('Unit')]
        neuron_data_session = neuron_data_session[unit_cols]
        
        # Map top 10 neurons to Unit column names
        topx_unit_cols = [f'Unit{i+1:04d}' for i in top_x_neurons if f'Unit{i+1:04d}' in neuron_data_session.columns]
        
        # Filter to only top 10 neurons
        neuron_data_top10 = neuron_data_session[topx_unit_cols]
        
        # Compute correlation matrix
        corr_matrix = neuron_data_top10.corr()
        
        # Create ordering based on reversed top 10 neurons
        unit_order = [f'Unit{i+1:04d}' for i in reversed_order_topx if f'Unit{i+1:04d}' in corr_matrix.columns]
        unit_labels = [u.replace('Unit00', '') for u in unit_order]
        
        # Reorder the correlation matrix
        corr_matrix_ordered = corr_matrix.loc[unit_order, unit_order]
        
        # Add heatmap to subplot
        fig.add_trace(
            go.Heatmap(
                z=corr_matrix_ordered.values,
                x=unit_labels,
                y=unit_labels,
                colorscale='RdBu_r',
                zmin=-1,
                zmax=1,
                showscale=(idx == 0),
                colorbar=dict(
                    title="Pearson r",
                    x=1.02
                ) if idx == 0 else None,
                hovertemplate='X: %{x}<br>Y: %{y}<br>r: %{z:.3f}<extra></extra>'
            ),
            row=row,
            col=col
        )

        fig.add_annotation(
            text=f"S{session_id.split('_')[0]}",
            xref=f"x{idx + 1}",
            yref=f"y{idx + 1}",
            x=7,
            y=19.5,
            xanchor="left",
            yanchor="top",
            showarrow=False,
            font=dict(size=10, color="black"),
            bgcolor="rgba(255,255,255,0.2)"
        )
        
        # Update axes - show tick labels for top 10
        fig.update_xaxes(showticklabels=True, tickangle=90, tickfont=dict(size=8), row=row, col=col)
        fig.update_yaxes(showticklabels=True, tickfont=dict(size=8), row=row, col=col)
        
    except Exception as e:
        print(f"Error processing session {session_id}: {e}")
        continue

# Update layout
fig.update_layout(
    title_text=f"Top 20 Most Weighted Neurons: Pearson Correlation Matrices Across Sessions {selected[0].split('_')[0]} to {selected[-1].split('_')[0]}<br>(Assembly012 ordered by weights)",
    title_font_size=16,
    width=300 * n_cols,
    height=300 * n_rows,
    showlegend=False,
    #title=dict(y=0.9),
)


fig.show()

print(f"Created {n_sessions} correlation heatmaps (10x10) in {n_rows}x{n_cols} grid")
print(f"Top 10 most weighted neurons: {top_x_neurons.values}")

Created 8 correlation heatmaps (10x10) in 2x4 grid
Top 10 most weighted neurons: [21 56 74 72  2 35 65  7 62  8 36 68 23 14 24 22 48 15 11 50]


In [47]:
top_ns = [20]
session_ids_all = fr_track_selected.index.get_level_values("session_id").unique()
fr_used = fr_z_all_sess

def mean_offdiag_corr(corr_df: pd.DataFrame) -> float:
    k = corr_df.shape[0]
    if k < 2:
        return np.nan
    iu = np.triu_indices(k, k=1)
    vals = corr_df.to_numpy()[iu]
    return float(np.nanmean(vals))

# second y-axis for ensemble mean
fig = make_subplots(specs=[[{"secondary_y": True}]])

for top_n in top_ns:
    top_n_neurons = neurons_ordered[-top_n:]
    rows = []

    for session_id in session_ids_all:
        try:
            neuron_data_session = fr_used.xs(session_id, level=2) 

            unit_cols = [
                c for c in neuron_data_session.columns
                if isinstance(c, str) and c.startswith("Unit")
            ]
            neuron_data_session = neuron_data_session[unit_cols]

            top_unit_cols = [
                f"Unit{i+1:04d}" for i in top_n_neurons
                if f"Unit{i+1:04d}" in neuron_data_session.columns
            ]

            neuron_data_top = neuron_data_session[top_unit_cols]
            print(neuron_data_top.keys())
            corr = neuron_data_top.corr(method="pearson")
            avg_corr = mean_offdiag_corr(corr)

            rows.append((session_id, avg_corr))
        except Exception:
            rows.append((session_id, np.nan))

    df_avg = pd.DataFrame(rows, columns=["session_id", "avg_pairwise_corr"])

    fig.add_trace(
        go.Scatter(
            x=df_avg["session_id"],
            y=df_avg["avg_pairwise_corr"],
            mode="lines+markers",
            name=f"top {top_n} neuron correlation",
        ),
        secondary_y=False,
    )

# ensemble mean per session (one trace on the right y-axis)
ens_mean = (
    ensamble_proj["Assembly012"]
    .groupby(level=0)
    .mean()
    .reindex(session_ids_all)
)

fig.add_trace(
    go.Scatter(
        x=ens_mean.index,
        y=ens_mean.values,
        mode="lines+markers",
        name="Ensemble mean (Assembly012)",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Average pairwise Pearson correlation across sessions",
    width=900,
    height=450,
)

fig.update_xaxes(title_text="Session")

fig.update_yaxes(
    title_text="Average pairwise correlation (Pearson r)",
    range=[-1, 1],
    secondary_y=False,
)

fig.update_yaxes(
    title_text="Ensemble activation (mean)",
    secondary_y=True,
)

fig.show()


Index(['Unit0022', 'Unit0057', 'Unit0075', 'Unit0073', 'Unit0003', 'Unit0036',
       'Unit0066', 'Unit0008', 'Unit0063', 'Unit0009', 'Unit0037', 'Unit0069',
       'Unit0024', 'Unit0015', 'Unit0025', 'Unit0023', 'Unit0049', 'Unit0016',
       'Unit0012', 'Unit0051'],
      dtype='object')
Index(['Unit0022', 'Unit0057', 'Unit0075', 'Unit0073', 'Unit0003', 'Unit0036',
       'Unit0066', 'Unit0008', 'Unit0063', 'Unit0009', 'Unit0037', 'Unit0069',
       'Unit0024', 'Unit0015', 'Unit0025', 'Unit0023', 'Unit0049', 'Unit0016',
       'Unit0012', 'Unit0051'],
      dtype='object')
Index(['Unit0022', 'Unit0057', 'Unit0075', 'Unit0073', 'Unit0003', 'Unit0036',
       'Unit0066', 'Unit0008', 'Unit0063', 'Unit0009', 'Unit0037', 'Unit0069',
       'Unit0024', 'Unit0015', 'Unit0025', 'Unit0023', 'Unit0049', 'Unit0016',
       'Unit0012', 'Unit0051'],
      dtype='object')
Index(['Unit0022', 'Unit0057', 'Unit0075', 'Unit0073', 'Unit0003', 'Unit0036',
       'Unit0066', 'Unit0008', 'Unit0063', 'Unit

In [19]:
top_ns = [10, 20, 77]
fr_used = fr_z_all_sess.copy()

# 2x2 layout: correlations on first row, firing rates on second row
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Pearson corr (unweighted)", "Pearson corr (weighted)",
        "Mean firing rate (unweighted)", "Mean firing rate (weighted)"
    ),
    shared_yaxes=False,
)

weights = ensambles["Assembly012"]
try:
    weights.index = [f"Unit{int(i)+1:04d}" for i in weights.index]  # align to FR column naming
except Exception as e:
    print(f"Error setting weights index: {e}")



# pearson correlations for single neurons weighted and unweightes
for top_n in top_ns:
    top_n_neurons = neurons_ordered[-top_n:]
    rows_unw, rows_w = [], []

    for session_id in session_ids_all:
        neuron_data_session = fr_used.xs(session_id, level=2)

        unit_cols = [
            c for c in neuron_data_session.columns
            if isinstance(c, str) and c.startswith("Unit")
        ]
        neuron_data_session = neuron_data_session[unit_cols]

        top_unit_cols = [
            f"Unit{i+1:04d}" for i in top_n_neurons
            if f"Unit{i+1:04d}" in neuron_data_session.columns
        ]

        neuron_data_top = neuron_data_session[top_unit_cols]

        corr_unw = neuron_data_top.corr(method="pearson")
        rows_unw.append((session_id, mean_offdiag_corr(corr_unw)))

        w = weights.loc[top_unit_cols]
        neuron_data_top_w = neuron_data_top.mul(w, axis=1)
        corr_w = neuron_data_top_w.corr(method="pearson")
        rows_w.append((session_id, mean_offdiag_corr(corr_w)))

    df_unw = pd.DataFrame(rows_unw, columns=["session_id", "avg_corr"])
    df_w = pd.DataFrame(rows_w, columns=["session_id", "avg_corr"])

    fig.add_trace(
        go.Scatter(
            x=df_unw["session_id"],
            y=df_unw["avg_corr"],
            mode="lines+markers",
            name=f"top {top_n}",
            legendgroup=f"top{top_n}",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=df_w["session_id"],
            y=df_w["avg_corr"],
            mode="lines+markers",
            name=f"top {top_n}",
            legendgroup=f"top{top_n}",
            showlegend=False,  # avoid duplicated legend entries
        ),
        row=1,
        col=2,
    )


# mean fr
fr_units = fr_used.loc[:, fr_used.columns.str.startswith("Unit")]
# fr_units = fr.loc[:, fr.columns.str.startswith("Unit")]
fr_units = fr_units[fr_units.index.get_level_values(2) != 10]

w_all = ensambles["Assembly012"].copy()
w_all = w_all.reindex(fr_units.columns)
fr_units_weighted = fr_units.mul(w_all, axis=1)

session_means_unweighted = (
    fr_units
    .groupby(level=2)
    .mean()
    .mean(axis=1)
)

session_means_weighted = (
    fr_units_weighted
    .groupby(level=2)
    .mean()
    .mean(axis=1)
)

fig.add_scatter(
    x=session_means_unweighted.index,
    y=session_means_unweighted.values,
    mode="lines+markers",
    name="Mean FR unweighted",
    row=2,
    col=1,
)

fig.add_scatter(
    x=session_means_weighted.index,
    y=session_means_weighted.values,
    mode="lines+markers",
    name="Mean FR weighted",
    row=2,
    col=2,
)


fig.update_layout(
    title="Session-wise Pearson correlations (top row) and mean firing rates (bottom row)",
    width=1200,
    height=800,
)

fig.update_xaxes(title_text="Session", row=1, col=1)
fig.update_xaxes(title_text="Session", row=1, col=2)
fig.update_xaxes(title_text="Session", row=2, col=1)
fig.update_xaxes(title_text="Session", row=2, col=2)

fig.update_yaxes(title_text="Avg pairwise Pearson r", row=1, col=1)
fig.update_yaxes(title_text="Avg pairwise Pearson r", row=1, col=2)
fig.update_yaxes(title_text="Mean firing rate (z-scored)", row=2, col=1)
fig.update_yaxes(title_text="Mean firing rate (z-scored)", row=2, col=2)

fig.update_yaxes(range=[-1, 1], row=1, col=1)
fig.update_yaxes(range=[-1, 1], row=1, col=2)

fig.show()


In [20]:
s = '2024-12-09_17-45'
neuron_data_s15 = fr_track.xs(s, level=2)[fr_track.xs(s, level=2)['trial_id'] == 23]
unit_cols = [col for col in neuron_data_s15.columns if isinstance(col, str) and col.startswith('Unit')]
neuron_data_s15 = neuron_data_s15[unit_cols]

corr_matrix = neuron_data_s15.corr()

reversed_order = neurons_ordered[::-1]
unit_order = [f'Unit{i+1:04d}' for i in reversed_order if f'Unit{i+1:04d}' in corr_matrix.columns]

corr_matrix_ordered = corr_matrix.loc[unit_order, unit_order]

fig = px.imshow(
    corr_matrix_ordered,
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    labels=dict(x="Neurons", y="Neurons", color="Pearson r"),
    title=f"Neuron-Neuron Pearson Correlation Matrix (Session {s}, Assembly012 ordering reversed)"
)

fig.update_xaxes(tickangle=90, tickfont=dict(size=6))
fig.update_yaxes(tickfont=dict(size=6))
fig.update_layout(width=900, height=900)

fig.show()

In [21]:
# selectiung only the data of ens_data in the cue zone and right before the reward zones (-95 to 10; 148-151; 168-171)
ens_data_selected = ens_data[(ens_data['from_position_bin'] >= (-95)) & (ens_data['from_position_bin'] <= 10)]
# adding reward zones with bins 148-151; 168-171
ens_data_selected = pd.concat([
    ens_data_selected,
    ens_data[(ens_data['from_position_bin'] >= 148) & (ens_data['from_position_bin'] <= 151)],
    ens_data[(ens_data['from_position_bin'] >= 168) & (ens_data['from_position_bin'] <= 171)]
])

In [22]:
df = ens_data

# If session_id is in the index, bring it back as a column
if "session_id" not in df.columns:
    if isinstance(df.index, pd.MultiIndex) and "session_id" in df.index.names:
        df = df.reset_index()
    elif df.index.name == "session_id":
        df = df.reset_index()

trial_avg = (
    df.groupby(["session_id", "from_position_bin", "cue"], as_index=False)["Assembly012"]
      .mean()
      .rename(columns={"Assembly012": "Assembly012_mean"})
)

trial_avg = trial_avg[trial_avg["cue"].isin([1, 2])].copy()

# sorting x-axis values
pos_numeric = pd.to_numeric(trial_avg["from_position_bin"], errors="coerce")
if pos_numeric.notna().any():
    trial_avg["_pos"] = pos_numeric
    x_order = np.sort(trial_avg["_pos"].dropna().unique())
    # keep a numeric x for correct ordering/spacing
    trial_avg["_x"] = trial_avg["_pos"]
    x_ticks = x_order
    x_ticktext = [str(int(x)) if float(x).is_integer() else str(x) for x in x_order]
else:
    # fallback: treat as ordered categorical by appearance
    trial_avg["_x"] = trial_avg["from_position_bin"].astype(str)
    x_order = list(pd.unique(trial_avg["_x"]))
    x_ticks = x_order
    x_ticktext = x_order

# Ridgeline plot parameters
sessions = sorted(trial_avg["session_id"].unique())
dy = 0.8          # vertical spacing between sessions
cue_offset = 0.02 # vertical separation between cue 1 and cue 2 within a session
amp = 0.2

fig = go.Figure()

for i, s in enumerate(sessions):
    base = i * dy

    for cue, off in [(1, -cue_offset), (2, +cue_offset)]:
        sub = trial_avg[(trial_avg["session_id"] == s) & (trial_avg["cue"] == cue)].copy()

        # Ensure every x bin exists, fill missing with 0 (or np.nan if you prefer gaps)
        if pos_numeric.notna().any():
            sub = sub.set_index("_x").reindex(x_order).reset_index()
        else:
            sub = sub.set_index("_x").reindex(x_order).reset_index()

        y = sub["Assembly012_mean"].fillna(0.0).to_numpy()
        y_ridge = base + off + amp * y

        fig.add_trace(
            go.Scatter(
                x=sub["_x"],
                y=y_ridge,
                mode="lines",
                line=dict(width=1, color= cue == 1 and "orange" or "purple"),
                fill= None,
                name=f"session {s} | cue {cue}",
                hovertemplate=(
                    "session=%{customdata[0]}<br>"
                    "cue=%{customdata[1]}<br>"
                    "from_position_bin=%{x}<br>"
                    "mean(Assembly012)=%{customdata[2]:.4f}<extra></extra>"
                ),
                customdata=np.c_[np.full(len(sub), s), np.full(len(sub), cue), y],
                showlegend=True,
            )
        )



#Layout polish
fig.update_layout(
    title="Ridgeline: mean(Assembly012) per from_position_bin, one ridge per session",
    xaxis_title="from_position_bin",
    yaxis_title="session_id (stacked ridges)",
    # hovermode="x",
    height=max(450, 40 * len(sessions) + 200),
)

# Put session labels on the y-axis at each session baseline
fig.update_yaxes(
    tickmode="array",
    tickvals=[i * dy for i in range(len(sessions))],
    ticktext=[str(s) for s in sessions],
)

fig.add_vrect(x0=-80, x1=25,  fillcolor="orange", opacity=0.2, layer="below", line_width=0)
fig.add_vrect(x0=50,  x1=110, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)
fig.add_vrect(x0=170, x1=230, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)


if pos_numeric.notna().any():
    fig.update_xaxes(
        tickmode="array",
        ticktext=x_ticktext,
        showgrid=False, zeroline=False
    )

fig.show()


In [23]:
df = ens_data
need_reset = False
if isinstance(df.index, pd.MultiIndex):
    need_reset = any(name in df.index.names for name in ["session_id", "trial_id"])
elif df.index.name in ["session_id", "trial_id"]:
    need_reset = True

if need_reset:
    df = df.reset_index()

# session selector
# session_to_plot = df["session_id"].dropna().unique()[0]  # first available session as default
session_to_plot = '2024-12-09_17-45'

df_s = df.loc[df["session_id"] == session_to_plot].copy()
#df_s = df_s[df_s["cue"].isin([1, 2])].copy()

# If "cue" is constant per trial, this will keep it; if not, it will take the first observed cue per trial
trial_cue = (
    df_s.groupby("trial_id", as_index=True)["cue"]
        .agg(lambda x: x.dropna().iloc[0] if len(x.dropna()) else np.nan)
)

# Aggregate within trial and position bin
trial_avg = (
    df_s.groupby(["trial_id", "from_position_bin"], as_index=False)["Assembly012"]
        .mean()
        .rename(columns={"Assembly012": "Assembly012_mean"})
)


# Build ordered numeric x if possible
pos_numeric = pd.to_numeric(trial_avg["from_position_bin"], errors="coerce")
if pos_numeric.notna().any():
    trial_avg["_x"] = pos_numeric
    x_order = np.sort(trial_avg["_x"].dropna().unique())
    x_tickvals = x_order
    x_ticktext = [str(int(x)) if float(x).is_integer() else str(x) for x in x_order]
else:
    trial_avg["_x"] = trial_avg["from_position_bin"].astype(str)
    x_order = list(pd.unique(trial_avg["_x"]))
    x_tickvals = x_order
    x_ticktext = x_order


# Ridgeline parameters
trials = sorted(trial_avg["trial_id"].unique())
dy = 0.8      #between trials
amp = 0.05    # amplitude scaling

fig = go.Figure()

for i, t in enumerate(trials):
    base = i * dy

    sub = trial_avg.loc[trial_avg["trial_id"] == t, ["_x", "Assembly012_mean"]].copy()
    sub = sub.set_index("_x").reindex(x_order).reset_index()

    y = sub["Assembly012_mean"].fillna(0.0).to_numpy()
    y_ridge = base + amp * y

    cue_t = trial_cue.get(t, np.nan)
    line_color = "orange" if cue_t == 1 else "purple"

    fig.add_trace(
        go.Scatter(
            x=sub["_x"],
            y=y_ridge,
            mode="lines",
            line=dict(width=1, color=line_color),
            name=f"trial {t}",
            hovertemplate=(
                "session=%{customdata[0]}<br>"
                "trial_id=%{customdata[1]}<br>"
                "cue=%{customdata[2]}<br>"
                "from_position_bin=%{x}<br>"
                "mean(Assembly012)=%{customdata[3]:.4f}<extra></extra>"
            ),
            customdata=np.c_[
                np.full(len(sub), session_to_plot),
                np.full(len(sub), t),
                np.full(len(sub), cue_t),
                y,
            ],
            showlegend=False,
        )
    )

# Y-axis
fig.update_yaxes(
    tickmode="array",
    tickvals=[i * dy for i in range(len(trials))],
    ticktext=[str(t) for t in trials],
    title_text="trial_id",
)

fig.update_layout(
    title=f"Ridgeline: mean(Assembly012) per from_position_bin, one ridge per trial (session {session_to_plot})",
    xaxis_title="from_position_bin",
    height=max(500, 30 * len(trials) + 250),
)

fig.update_xaxes(
    tickmode="array",
    showgrid=False,
    zeroline=False,
)

fig.add_vrect(x0=-80, x1=25,  fillcolor="orange", opacity=0.2, layer="below", line_width=0)
fig.add_vrect(x0=50,  x1=110, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)
fig.add_vrect(x0=170, x1=230, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)

fig.show()


In [24]:
Assembly = 'Assembly012'
i = '2024-12-06_16-49'
s1 = ens_data.xs(i, level='session_id')
s1_avg = (s1.groupby(['from_position_bin','cue'], as_index=False)[Assembly].mean())
title = f'Average Ensemble Activityby Position Across Trials in Session {i}'
fig = px.line(
    s1_avg,
    x='from_position_bin',
    y=Assembly,
    color='cue',
    color_discrete_map={1: 'orange', 2: 'purple'},
    markers=True,
    title=title,
    range_y=[-1, 1.5]
)
fig.add_vrect(x0=-80, x1=25,  fillcolor="orange", opacity=0.3, layer="below", line_width=0)
fig.add_vrect(x0=50,  x1=110, fillcolor="grey",   opacity=0.4, layer="below", line_width=0)
fig.add_vrect(x0=170, x1=230, fillcolor="grey",   opacity=0.2, layer="below", line_width=0)

fig.show()


# plotting the avg. activity for each trial in session 14 in a scatter plot
s1 = ens_data_selected.xs(i, level='session_id')
s1_avg = s1.groupby('trial_id').mean().reset_index()
s1_avg['trial_id'] = s1_avg['trial_id'] - 1
title = f'Average Ensemble Activity in the Cue Zones and Before Reward Zones by Trial for Session {i}'
# fig = px.line(s1_avg, x='trial_id', y='Assembly012', title=title, range_y=[-1,2])
fig = px.line(
    s1_avg,
    x='trial_id',
    y=Assembly,
    color='cue',
    color_discrete_map={1: 'orange', 2: 'purple'},
    markers=True,
    title=title,
    range_y=[-1, 2]
)
fig.show()

In [25]:
s = '2024-12-06_16-49' #session
i = 19 #trial
s1_trial1 = ens_data.xs((s, i), level=('session_id', 'trial_id'))
title = f'Firing Rate by Position for Trial {i} in Session {s}'

fig = px.line(s1_trial1, x='from_position_bin', y='Assembly012', color = 'cue', title=title)
fig.add_vrect(x0=-80, x1=25, fillcolor="orange", opacity=0.3, layer="below", line_width=0)
fig.add_vrect(x0=50, x1=110, fillcolor="grey", opacity=0.4, layer="below", line_width=0)
fig.add_vrect(x0=170, x1=230, fillcolor="grey", opacity=0.2, layer="below", line_width=0)
fig.show()

print(f'Trial {i-1} is a Cue {s1_trial1["cue"]} trial in Session {s}')

Trial 18 is a Cue session_id        trial_id
2024-12-06_16-49  19.0        1.0
                  19.0        1.0
                  19.0        1.0
                  19.0        1.0
                  19.0        1.0
                             ... 
                  19.0        1.0
                  19.0        1.0
                  19.0        1.0
                  19.0        1.0
                  19.0        1.0
Name: cue, Length: 428, dtype: float64 trial in Session 2024-12-06_16-49


# Distance Score according to Sun et al. (2025)

- D = |A_near − A_far|/max(A_near, A_far)
- Pearson correlation

to compute differences between near and far trials over sessions for: 
- Top n neurons of ensemble
- All neurons weighted
- Ensmeble

In [26]:
# dropping all neurons that are not in top n
n = 20
top_x_neurons = neurons_ordered[-n:]
top_x_neurons
top_unit_cols = [
    f"Unit{i+1:04d}" for i in top_x_neurons
    if f"Unit{i+1:04d}" in fr_track.columns
]
mask = (~fr_track.columns.str.startswith("Unit")) | (fr_track.columns.isin(top_unit_cols))


mask
fr_top_n = fr_track.loc[:, mask]
fr_top_n

from_position_bin  trial_id  \
paradigm_id animal_id session_id       entry_id                                
1100        6         2024-11-14_16-40 0                    -169.0         1   
                                       1                    -169.0         2   
                                       2                    -169.0         3   
                                       3                    -169.0         4   
                                       4                    -169.0         5   
...                                                            ...       ...   
                      2025-01-27_13-39 65488                 262.0       123   
                                       65489                 262.0       124   
                                       65490                 262.0       128   
                                       65491                 262.0       135   
                                       65492                 262.0       139   

                                                 Unit0003  Unit0008  Unit0009  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0              0.0       0.0       0.0   
                                       1              0.0       0.0       0.0   
                                       2              0.0       0.0       0.0   
                                       3              0.0       0.0       0.0   
                                       4              0.0       0.0       0.0   
...                                                   ...       ...       ...   
                      2025-01-27_13-39 65488          0.0       0.0       0.0   
                                       65489          0.0       0.0       0.0   
                                       65490          0.0       0.0       0.0   
                                       65491          0.0       0.0       0.0   
                                       65492          0.0       0.0       0.0   

                                                  Unit0012   Unit0015  \
paradigm_id animal_id session_id       entry_id                         
1100        6         2024-11-14_16-40 0         20.000000  53.333332   
                                       1         20.000000  45.000000   
                                       2          3.333333  45.000000   
                                       3          1.666667  45.000000   
                                       4          6.666667  43.333332   
...                                                    ...        ...   
                      2025-01-27_13-39 65488      0.000000  25.000000   
                                       65489      0.000000  25.000000   
                                       65490      0.000000  12.500000   
                                       65491      0.000000  50.000000   
                                       65492      0.000000  12.500000   

                                                 Unit0016   Unit0022  \
paradigm_id animal_id session_id       entry_id                        
1100        6         2024-11-14_16-40 0         0.000000  10.000000   
                                       1         1.666667   0.000000   
                                       2         0.000000   1.666667   
                                       3         1.666667   5.000000   
                                       4         0.000000   3.333333   
...                                                   ...        ...   
                      2025-01-27_13-39 65488     0.000000  12.500000   
                                       65489     0.000000   0.000000   
                                       65490     0.000000   0.000000   
                                       65491     0.000000   0.000000   
                                       65492     0.000000  12.500000   

                                                  Unit0023  ...  Un

In [27]:
# distance scoire function following sun et al
def get_distance_score(A_near, A_far):
    denom = np.maximum(A_near, A_far)
    out = np.zeros_like(denom, dtype=float)
    m = denom != 0
    out[m] = np.abs(A_near[m] - A_far[m]) / denom[m]
    return out

distance_scores_cue = []
distance_scores_r1 = []
distance_scores_r2  = []

for session_id in fr_track.index.get_level_values('session_id').unique():
    drop_session_data = fr_track.xs(session_id, level='session_id')
    
    a_near_cue = []
    a_near_r1 = []
    a_near_r2 = []

    a_far_cue = []
    a_far_r1 = []
    a_far_r2 = []

    # max for each trial and append to near or far depending on the cue
    for trial_id in drop_session_data['trial_id'].values.unique():
        trial_data = drop_session_data[drop_session_data['trial_id'] == trial_id]  # ['Assembly012']
        mask_cols = trial_data.columns.str.startswith("Unit")
        a_trial_cue = trial_data.loc[trial_data['from_position_bin'].between(-80, 25), mask_cols].max().max()
        a_trial_r1 = trial_data.loc[trial_data['from_position_bin'].between(50, 110), mask_cols].max().max()
        a_trial_r2 = trial_data.loc[trial_data['from_position_bin'].between(170, 230), mask_cols].max().max()
        if (trial_data['cue'] == 1).all():
            a_near_cue.append(a_trial_cue)
            a_near_r1.append(a_trial_r1)
            a_near_r2.append(a_trial_r2)
        elif (trial_data['cue'] == 2).all():
            a_far_cue.append(a_trial_cue)
            a_far_r1.append(a_trial_r1)
            a_far_r2.append(a_trial_r2)
        else:
            print(f"Warning: trial {trial_id} in session {session_id} has mixed cues. Skipping.")
    distance_score_cue = get_distance_score(np.mean(a_near_cue), np.mean(a_far_cue))
    distance_score_r1 = get_distance_score(np.mean(a_near_r1), np.mean(a_far_r1))
    distance_score_r2 = get_distance_score(np.mean(a_near_r2), np.mean(a_far_r2))
    

    distance_scores_cue.append({'session_id': session_id, 'distance_score': distance_score_cue})
    distance_scores_r1.append({'session_id': session_id, 'distance_score': distance_score_r1})
    distance_scores_r2.append({'session_id': session_id, 'distance_score': distance_score_r2})


distance_scores_cue
    
        # distance_score = get_distance_score(trial_data)
        # fr_track.loc[(slice(None), trial_id), 'distance_score'] = distance_score

[{'session_id': '2024-11-14_16-40', 'distance_score': array(0.00963489)},
 {'session_id': '2024-11-15_15-48', 'distance_score': array(0.01491094)},
 {'session_id': '2024-11-20_17-46', 'distance_score': array(0.05494505)},
 {'session_id': '2024-11-21_17-22', 'distance_score': array(0.02857143)},
 {'session_id': '2024-11-25_16-25', 'distance_score': array(0.00246305)},
 {'session_id': '2024-11-26_16-39', 'distance_score': array(0.0253035)},
 {'session_id': '2024-11-28_17-41', 'distance_score': array(0.05655406)},
 {'session_id': '2024-12-02_16-09', 'distance_score': array(0.0675257)},
 {'session_id': '2024-12-03_16-23', 'distance_score': array(0.11969062)},
 {'session_id': '2024-12-04_18-06', 'distance_score': array(0.02737542)},
 {'session_id': '2024-12-06_16-49', 'distance_score': array(0.06759129)},
 {'session_id': '2024-12-09_17-45', 'distance_score': array(0.07523148)},
 {'session_id': '2024-12-10_17-20', 'distance_score': array(0.10775566)},
 {'session_id': '2024-12-12_16-13', 'dis

In [28]:
def get_distance_score(A_near, A_far):
    denom = np.maximum(A_near, A_far)
    out = np.zeros_like(denom, dtype=float)
    m = denom != 0
    out[m] = np.abs(A_near[m] - A_far[m]) / denom[m]
    return out

windows = {
    'start-end': (-169, -80),
    "cue": (-80, 25),
    # "r1":  (50, 110),
    # "r2":  (170, 230),
    'r1+r2': (25, 230),
}

unit_prefix = "Unit"
results = []

for session_id in fr_top_n.index.get_level_values("session_id").unique():
    drop_session_data = fr_top_n.xs(session_id, level="session_id")
    unit_cols = drop_session_data.columns[drop_session_data.columns.str.startswith(unit_prefix)]

    near = {k: [] for k in windows}
    far  = {k: [] for k in windows}

    for trial_id in drop_session_data["trial_id"].unique():
        trial_data = drop_session_data[drop_session_data["trial_id"] == trial_id]

        if (trial_data["cue"] == 1).all():
            group = near
        elif (trial_data["cue"] == 2).all():
            group = far
        else:
            continue

        for wname, (lo, hi) in windows.items():
            s = (
                trial_data
                .loc[trial_data["from_position_bin"].between(lo, hi), unit_cols]
                .mean(axis=0)   # mean activation per unit
            )
            group[wname].append(s)

    for wname in windows:
        n1 = len(near[wname])
        n2 = len(far[wname])
        n_pair = min(n1, n2)

        #  distance scores according to Sun et al. 
        if n1 == 0 or n2 == 0:
            dist = pd.Series(np.nan, index=unit_cols)
        else:
            A_near = pd.concat(near[wname], axis=1).mean(axis=1)
            A_far  = pd.concat(far[wname],  axis=1).mean(axis=1)
            dist = pd.Series(
                get_distance_score(A_near.to_numpy(), A_far.to_numpy()),
                index=unit_cols
            )

        # Pearson correlaation
        if n_pair < 2:
            pearson_r = pd.Series(np.nan, index=unit_cols)
            pearson_p = pd.Series(np.nan, index=unit_cols)
        else:
            X = pd.concat(near[wname][:n_pair], axis=1)
            Y = pd.concat(far[wname][:n_pair],  axis=1)

            r_vals = np.empty(len(unit_cols))
            p_vals = np.empty(len(unit_cols))

            for i, u in enumerate(unit_cols):
                x = X.loc[u].to_numpy()
                y = Y.loc[u].to_numpy()

                if np.allclose(x, x[0]) or np.allclose(y, y[0]):
                    r_vals[i] = np.nan
                    p_vals[i] = np.nan
                else:
                    r_vals[i], p_vals[i] = pearsonr(x, y)

            pearson_r = pd.Series(r_vals, index=unit_cols)
            pearson_p = pd.Series(p_vals, index=unit_cols)

        tmp = pd.DataFrame({
            "unit": unit_cols,
            "distance_score": dist.values,
            "pearson_r": pearson_r.values,
            "pearson_p": pearson_p.values,
            "session_id": session_id,
            "window": wname,
            "n_trials_cue1": n1,
            "n_trials_cue2": n2,
            "n_pairs_used": n_pair,
        })

        results.append(tmp)

distance_metrics_per_unit = (
    pd.concat(results, ignore_index=True)
      .set_index(["session_id", "window", "unit"])
      .sort_index()
)

distance_metrics_per_unit


distance_score  pearson_r  pearson_p  \
session_id       window    unit                                             
2024-11-14_16-40 cue       Unit0003        0.336319  -0.180217   0.220298   
                           Unit0008        0.000000        NaN        NaN   
                           Unit0009        0.640106   0.257124   0.077691   
                           Unit0012        0.272207   0.180020   0.220812   
                           Unit0015        0.031262  -0.085667   0.562631   
...                                             ...        ...        ...   
2025-01-27_13-39 start-end Unit0063        0.052631  -0.031638   0.789011   
                           Unit0066        0.025279   0.054519   0.644548   
                           Unit0069        0.000887   0.242451   0.037406   
                           Unit0073        0.379929   0.124005   0.292507   
                           Unit0075        0.091203   0.105304   0.371896   

                                     n_trials_cue1  n_trials_cue2  \
session_id       window    unit                                     
2024-11-14_16-40 cue       Unit0003             54             48   
                           Unit0008             54             48   
                           Unit0009             54             48   
                           Unit0012             54             48   
                           Unit0015             54             48   
...                                            ...            ...   
2025-01-27_13-39 start-end Unit0063             74             78   
                           Unit0066             74             78   
                           Unit0069             74             78   
                           Unit0073             74             78   
                           Unit0075             74             78   

                                     n_pairs_used  
session_id       window    unit                    
2024-11-14_16-40 cue       Unit0003            48  
                           Unit0008            48  
                           Unit0009            48  
                           Unit0012            48  
                           Unit0015            48  
...                                           ...  
2025-01-27_13-39 start-end Unit0063            74  
                           Unit0066            74  
                           Unit0069            74  
                           Unit0073            74  
                           Unit0075            74  

[1500 rows x 6 columns]

In [29]:
df = distance_metrics_per_unit.reset_index()

# Ensure consistent row ordering
#window_order = ["cue", "r1", "r2"]
window_order = ['start-end', 'cue', 'r1+r2'] 
df = df[df["window"].isin(window_order)].copy()
df["window"] = pd.Categorical(df["window"], categories=window_order, ordered=True)

# Pick up to 10 sessions
session_ids = (
    df["session_id"]
    .drop_duplicates()
    .tolist()
)[:30]

n_rows = 3
n_cols = 30

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    row_titles=window_order,
    column_titles=[str(s) for s in session_ids] + [""] * (n_cols - len(session_ids)),
    horizontal_spacing=0.02,
    vertical_spacing=0.06,
)

x_range = [-1.0, 1.0]   
y_range = [0.0, 1.0] 

for c, session_id in enumerate(session_ids, start=1):
    d_sess = df[df["session_id"] == session_id]

    for r, w in enumerate(window_order, start=1):
        d = d_sess[d_sess["window"] == w]

        # Filter invalid points
        d = d[np.isfinite(d["pearson_r"]) & np.isfinite(d["distance_score"])]

        fig.add_trace(
            go.Scatter(
                x=d["pearson_r"],
                y=d["distance_score"],
                mode="markers",
                marker=dict(size=5, opacity=0.7),
                hovertemplate=(
                    "session=%{customdata[0]}<br>"
                    "window=%{customdata[1]}<br>"
                    "unit=%{customdata[2]}<br>"
                    "r=%{x:.3f}<br>"
                    "dist=%{y:.3f}<extra></extra>"
                ),
                customdata=np.stack(
                    [d["session_id"].astype(str), d["window"].astype(str), d["unit"].astype(str)],
                    axis=1
                ),
                showlegend=False,
            ),
            row=r,
            col=c,
        )

        # Per-subplot axes formatting
        fig.update_xaxes(range=x_range, row=r, col=c, zeroline=True)
        fig.update_yaxes(range=y_range, row=r, col=c, zeroline=True)

# Axis labels: only on outer plots to reduce clutter
for c in range(1, len(session_ids) + 1):
    fig.update_xaxes(title_text="Pearson r", row=3, col=c)
for r in range(1, 4):
    fig.update_yaxes(title_text="Distance score", row=r, col=1)

fig.update_layout(
    height=800,
    width=14400,
    margin=dict(l=5, r=5, t=80, b=40),
    title_text="Per-unit distance score vs cue1–cue2 Pearson correlation (trial-wise mean activation)",
)

fig.show()


In [51]:
fr_track

from_position_bin  trial_id  \
paradigm_id animal_id session_id       entry_id                                
1100        6         2024-11-14_16-40 0                    -169.0         1   
                                       1                    -169.0         2   
                                       2                    -169.0         3   
                                       3                    -169.0         4   
                                       4                    -169.0         5   
...                                                            ...       ...   
                      2025-01-27_13-39 65488                 262.0       123   
                                       65489                 262.0       124   
                                       65490                 262.0       128   
                                       65491                 262.0       135   
                                       65492                 262.0       139   

                                                 Unit0001   Unit0002  \
paradigm_id animal_id session_id       entry_id                        
1100        6         2024-11-14_16-40 0         1.666667   0.000000   
                                       1         0.000000   0.000000   
                                       2         0.000000   3.333333   
                                       3         3.333333   0.000000   
                                       4         1.666667   0.000000   
...                                                   ...        ...   
                      2025-01-27_13-39 65488     0.000000   0.000000   
                                       65489     0.000000   0.000000   
                                       65490     0.000000  12.500000   
                                       65491     0.000000   0.000000   
                                       65492     0.000000   0.000000   

                                                 Unit0003  Unit0004  Unit0005  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0              0.0       0.0  1.666667   
                                       1              0.0       0.0  0.000000   
                                       2              0.0       0.0  0.000000   
                                       3              0.0       0.0  0.000000   
                                       4              0.0       0.0  1.666667   
...                                                   ...       ...       ...   
                      2025-01-27_13-39 65488          0.0       0.0  0.000000   
                                       65489          0.0       0.0  0.000000   
                                       65490          0.0      12.5  0.000000   
                                       65491          0.0       0.0  0.000000   
                                       65492          0.0       0.0  0.000000   

                                                 Unit0006  Unit0007  Unit0008  \
paradigm_id animal_id session_id       entry_id                                 
1100        6         2024-11-14_16-40 0              0.0       0.0       0.0   
                                       1              0.0       0.0       0.0   
                                       2              0.0       0.0       0.0   
                                       3              0.0       0.0       0.0   
                                       4              0.0       0.0       0.0   
...                                                   ...       ...       ...   
                      2025-01-27_13-39 65488          0.0       0.0       0.0   
                                       65489          0.0      25.0       0.0   
                                       65490          0.0       0.0       0.0   
                                       65491         12.5      25.0       0.0   
                                       65492 